# 2. Participant list (with dietary requirements)

One-page-per-few-participants PDF for the organisers: numbered name, career level and department, and a line with dietary requirements / allergies (for the catering). Synthetic data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Registration export (Google Form -> CSV). The repo ships a synthetic sample;
# point this to your own export to use it for a real event (keep that file out of git).
DATA = Path("sample_registrations.csv")
OUT = Path("output"); OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df.columns = df.columns.str.strip()           # the form export has trailing spaces in some headers
df = df[~df["Name"].isna()]                   # drop empty rows
df = df.drop_duplicates(subset="Email address", keep="first")   # people who registered twice
print(len(df), "unique registrations")

40 unique registrations


In [2]:
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import Paragraph, SimpleDocTemplate

base = getSampleStyleSheet()["Normal"]
s_name = ParagraphStyle("Name", parent=base, fontSize=10, textColor="crimson", leading=16, fontName="Helvetica-Bold")
s_affi = ParagraphStyle("Affi", parent=base, fontSize=10, leading=16, fontName="Helvetica-Oblique")
s_diet = ParagraphStyle("Diet", parent=base, fontSize=10, textColor="royalblue", spaceAfter=10, leading=16)

NO_DIET = "No dietary requirement"
ALLERGY_COL = "If any allergies, then please list them here"

def diet_line(row):
    parts = []
    if row["Dietary requirement"] != NO_DIET:
        parts.append(str(row["Dietary requirement"]))
    if pd.notna(row[ALLERGY_COL]):
        parts.append("allergies: " + str(row[ALLERGY_COL]))
    return " and ".join(parts) if parts else " "

story = []
for i, (_, row) in enumerate(df.iterrows(), start=1):
    story.append(Paragraph(escape(f"{i}. {row['Name']}"), s_name))
    story.append(Paragraph(escape(f"{row['Career level']} at {row['Department']}"), s_affi))
    story.append(Paragraph(escape(diet_line(row)), s_diet))

SimpleDocTemplate(str(OUT / "participant_list.pdf"), pagesize=letter).build(story)
print(len(df), "participants written")

40 participants written


## Catering summary

In [3]:
df["Dietary requirement"].value_counts()

Dietary requirement
No dietary requirement    31
Vegetarian                 6
Vegan                      2
Gluten-free                1
Name: count, dtype: int64